# 03 — Model Evaluation
Confusion matrix · SHAP · Per-class F1 · Cohen's Kappa · Top-3 hit rate


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    cohen_kappa_score, accuracy_score, f1_score
)

sns.set_theme(style='whitegrid')
FEATURES = ['Endurance','Strength','Speed','Flexibility','Cognitive Ability','Reflex']

df      = pd.read_csv('../data/combined_sports_data.csv').drop_duplicates().dropna()
le      = joblib.load('../models/label_encoder.pkl')
scaler  = joblib.load('../models/scaler.pkl')
ens     = joblib.load('../models/ensemble_model.pkl')
rf      = joblib.load('../models/random_forest_model.pkl')

y  = le.transform(df['Sport'])
X  = df[FEATURES].values
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
preds = ens.predict(X_test)

print(f'Accuracy    : {accuracy_score(y_test, preds):.4f}')
print(f'Macro F1    : {f1_score(y_test, preds, average="macro"):.4f}')
print(f'Cohen Kappa : {cohen_kappa_score(y_test, preds):.4f}')
print()
print(classification_report(y_test, preds, target_names=le.classes_, digits=3))

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax, linewidths=0.5, annot_kws={'size': 12})
ax.set_title('Confusion Matrix — Stacked Ensemble', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Per-class F1 ──
report  = classification_report(y_test, preds, target_names=le.classes_, output_dict=True)
class_f1 = {k: v['f1-score'] for k, v in report.items() if k in le.classes_}

fig, ax = plt.subplots(figsize=(9, 4))
colors  = ['#43A047' if v >= 0.95 else '#FFA726' if v >= 0.80 else '#EF5350' for v in class_f1.values()]
bars    = ax.bar(class_f1.keys(), class_f1.values(), color=colors, width=0.55, edgecolor='white')
for bar, v in zip(bars, class_f1.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.15)
ax.axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='0.90 target')
ax.set_title('Per-Class F1 Score', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('../results/per_class_f1.png', dpi=150)
plt.show()

In [ ]:
# ── Feature Importance (RF) ──
imp = rf.feature_importances_
idx = np.argsort(imp)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1565C0' if i == 0 else '#64B5F6' for i in range(len(FEATURES))]
bars   = ax.bar([FEATURES[i] for i in idx], imp[idx], color=colors, width=0.6, edgecolor='white')
for bar, v in zip(bars, imp[idx]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance'); plt.xticks(rotation=20, ha='right'); plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ── SHAP (install: pip install shap) ──
try:
    import shap
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test[:80])
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test[:80], feature_names=FEATURES,
                      class_names=list(le.classes_), plot_type='bar', show=False)
    plt.title('SHAP Mean |Value| — All Sports', fontsize=12)
    plt.tight_layout()
    plt.savefig('../results/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP plot saved.')
except ImportError:
    print('SHAP not installed. Run: pip install shap')

In [ ]:
# ── Top-3 hit rate ──
proba = ens.predict_proba(X_test)
top3  = np.argsort(proba, axis=1)[:, -3:]
hits  = sum(y_test[i] in top3[i] for i in range(len(y_test)))
print(f'Top-3 hit rate : {hits/len(y_test):.4f}  ({hits}/{len(y_test)})')
print(f'Cohen Kappa    : {cohen_kappa_score(y_test, preds):.4f}  (>0.8 = strong agreement)')